In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
(RESULTS / 'extraction-rasters').mkdir(parents=True, exist_ok=True)

In [ ]:
pseudo_codes_economies_142 = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')

from functools import lru_cache

def write_economy_supplement(path, values, profile, codes):
    with rio.open(path, 'w', **dict(profile, count=len(codes), dtype=values.dtype,
                                  nodata=None, compress='lzw')) as dst:
        dst.write(values)
        for band, code in enumerate(codes, 1):
            dst.set_band_description(band, str(int(code)))

@lru_cache(maxsize=None)
def economy_supplement(res):
    folder = ROOT / 'results' / 'economy-rasters'
    with rio.open(folder / f'economy-raster-eck4-{res}.tif') as coarse:
        codes = np.setdiff1d(pseudo_codes_economies_142['value'].unique(),
                            np.unique(coarse.read(1)))
        masks = np.zeros((len(codes), *coarse.shape), dtype=np.uint8)
        if len(codes):
            with rio.open(folder / 'economy-raster-eck4-1km.tif') as fine:
                if fine.crs != coarse.crs:
                    raise ValueError('Economy grids must use the same CRS')
                for row in range(0, fine.height, 256):
                    window = rio.windows.Window(0, row, fine.width, min(256, fine.height - row))
                    values = fine.read(1, window=window)
                    for band, code in enumerate(codes):
                        rr, cc = np.where(values == code)
                        if len(rr):
                            xx, yy = rio.transform.xy(fine.transform, rr + row, cc)
                            r, c = rio.transform.rowcol(coarse.transform, xx, yy)
                            masks[band, r, c] = 1
            if not masks.any(axis=(1, 2)).all():
                raise ValueError('An economy is also absent from the 1 km grid')
            write_economy_supplement(folder / f'economy-raster-eck4-{res}-supplement.tif',
                                     masks, coarse.profile, codes)
    return codes, masks.astype(bool)


def split_spatial_demand(total, raw_ratio):
    """Non-negative turnover/growth demands, preserving each draw's total.

    O/I remains an unbounded diagnostic in the material-flow model. Only its
    use as a demand share is bounded to [0, 1] here, before percentiles or spatial weights.
    """
    if total.shape != raw_ratio.shape:
        raise ValueError('Total extraction and turnover ratio must have identical shapes')
    if not np.isfinite(total).all() or (total < 0).any():
        raise ValueError('Total extraction must be finite and non-negative')
    active = total > 0
    if not np.isfinite(raw_ratio[active]).all():
        raise ValueError('Positive extraction has a non-finite turnover ratio')
    # A zero extraction demand has zero components, irrespective of its saved O/I.
    share = np.where(active, np.clip(raw_ratio, 0.0, 1.0), 0.0)
    turnover = share * total
    growth = (1.0 - share) * total
    if (turnover < 0).any() or (growth < 0).any():
        raise ValueError('Spatial demand components must be non-negative')
    if not np.allclose(turnover + growth, total, rtol=2e-7, atol=1e-6):
        raise ValueError('Turnover/growth split does not conserve extraction')
    return turnover, growth


@lru_cache(maxsize=None)
def spatial_demand_percentiles(economy, scenario, material):
    """Cache only annual percentile summaries, not the 10,000-draw arrays."""
    folder = Path(r'F:\aggregates-mining-biodiversity\results\material-flows') / economy / scenario
    ratio_name = {'sand': 'fine', 'gravel': 'coarse', 'limestone': 'RawMat_Ca'}[material]
    with np.load(folder / f'{material}_extraction.npz') as saved:
        total = saved['data']
    with np.load(folder / f'split_ratio_{ratio_name}.npz') as saved:
        raw_ratio = saved['data']
    try:
        turnover, growth = split_spatial_demand(total, raw_ratio)
    except ValueError as error:
        raise ValueError(f"{economy}/{scenario}/{material}: {error}") from error
    return (np.percentile(turnover, [2.5, 50, 97.5], axis=0),
            np.percentile(growth, [2.5, 50, 97.5], axis=0))


In [ ]:
def distribute_national_extraction_to_pixel(material, start, end, scenario, res='50km'):
    future = np.linspace(2023, 2050, 28)
    RESULTS = ROOT / "results"
    economy_raster_ = rio.open(RESULTS / 'economy-rasters' / f'economy-raster-eck4-{res}.tif')
    economy_raster = economy_raster_.read(1)
    supplement_codes, supplement_masks = economy_supplement(res)
    supplement_index = {int(code): band for band, code in enumerate(supplement_codes)}
    supplement_demand = np.zeros((3, len(supplement_codes), *economy_raster_.shape), dtype=np.float32)

    RESULTS = Path(r'F:\aggregates-mining-biodiversity\results')
    input_path_1 = RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{int(end-5)}-SSP2-{res}.tif'
    input_path_2 = RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{end}-SSP2-{res}.tif'

    global_building_minerals_start = rio.open(input_path_1).read(1)
    global_building_minerals_end = rio.open(input_path_2).read(1)

    extraction_demand_raster_for_stock_growth_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_growth_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_growth_u = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_u = np.zeros(shape=economy_raster_.shape, dtype="float32")

    with tqdm(range(1, 143), desc=f'distributing of {material} extraction demand in {start}-{end} of {scenario}') as pbar:
        for i in pbar:
            economy = pseudo_codes_economies_142[pseudo_codes_economies_142['value'] == i]['economies_142'].values[0]
            pbar.set_postfix_str(f'country={economy}')
            
            turnover_q, growth_q = spatial_demand_percentiles(economy, scenario, material)
            df_extraction_1 = pd.DataFrame(turnover_q.T, columns=['lower', 'mid', 'upper'], index=future)
            df_extraction_2 = pd.DataFrame(growth_q.T, columns=['lower', 'mid', 'upper'], index=future)

            extraction_for_stock_turnover_l = df_extraction_1.loc[start:end, 'lower'].sum()
            extraction_for_stock_turnover_m = df_extraction_1.loc[start:end, 'mid'].sum()
            extraction_for_stock_turnover_u = df_extraction_1.loc[start:end, 'upper'].sum()
            extraction_for_stock_growth_l = df_extraction_2.loc[start:end, 'lower'].sum()
            extraction_for_stock_growth_m = df_extraction_2.loc[start:end, 'mid'].sum()
            extraction_for_stock_growth_u = df_extraction_2.loc[start:end, 'upper'].sum()

            if i in supplement_index:
                band = supplement_index[i]
                mask = supplement_masks[band]
                turnover_weights = np.maximum(global_building_minerals_start[mask], 0).astype(np.float64)
                if turnover_weights.sum() == 0:
                    turnover_weights[:] = 1
                growth_weights = np.maximum(global_building_minerals_end[mask] - global_building_minerals_start[mask], 0).astype(np.float64)
                if growth_weights.sum() == 0:
                    growth_weights = turnover_weights.copy()
                turnover_weights /= turnover_weights.sum()
                growth_weights /= growth_weights.sum()
                for level, (turnover, growth) in enumerate(zip(
                    [extraction_for_stock_turnover_l, extraction_for_stock_turnover_m, extraction_for_stock_turnover_u],
                    [extraction_for_stock_growth_l, extraction_for_stock_growth_m, extraction_for_stock_growth_u],
                )):
                    supplement_demand[level, band][mask] = 1e3 * (turnover * turnover_weights + growth * growth_weights)
                continue

            mask_1 = economy_raster == i
            mask_2 = (economy_raster == i) & (global_building_minerals_end > global_building_minerals_start)

            total_1 = np.sum(global_building_minerals_start[mask_1])
            total_2 = np.sum(global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2])

            extraction_demand_raster_for_stock_turnover_l[mask_1] = 1e3 * extraction_for_stock_turnover_l * global_building_minerals_start[mask_1] / total_1
            extraction_demand_raster_for_stock_turnover_m[mask_1] = 1e3 * extraction_for_stock_turnover_m * global_building_minerals_start[mask_1] / total_1
            extraction_demand_raster_for_stock_turnover_u[mask_1] = 1e3 * extraction_for_stock_turnover_u * global_building_minerals_start[mask_1] / total_1

            if total_2 > 0:
                extraction_demand_raster_for_stock_growth_l[mask_2] = 1e3 * extraction_for_stock_growth_l * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
                extraction_demand_raster_for_stock_growth_m[mask_2] = 1e3 * extraction_for_stock_growth_m * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
                extraction_demand_raster_for_stock_growth_u[mask_2] = 1e3 * extraction_for_stock_growth_u * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
            else:
                # No positive growth cells: retain the national demand and use
                # existing building stocks, as for supplemental island economies.
                weights = np.maximum(global_building_minerals_start[mask_1], 0).astype(np.float64)
                if weights.sum() == 0:
                    weights[:] = 1
                weights /= weights.sum()
                for array, demand in zip(
                    [extraction_demand_raster_for_stock_growth_l,
                     extraction_demand_raster_for_stock_growth_m,
                     extraction_demand_raster_for_stock_growth_u],
                    [extraction_for_stock_growth_l, extraction_for_stock_growth_m,
                     extraction_for_stock_growth_u],
                ):
                    array[mask_1] = 1e3 * demand * weights
        
    extraction_demand_raster_l = extraction_demand_raster_for_stock_turnover_l + extraction_demand_raster_for_stock_growth_l
    extraction_demand_raster_m = extraction_demand_raster_for_stock_turnover_m + extraction_demand_raster_for_stock_growth_m
    extraction_demand_raster_u = extraction_demand_raster_for_stock_turnover_u + extraction_demand_raster_for_stock_growth_u


    RESULTS = ROOT / "results"
    output_path_L = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-L.tif'
    output_path_M = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-M.tif'
    output_path_H = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-H.tif'

    raster_set = [extraction_demand_raster_l, extraction_demand_raster_m, extraction_demand_raster_u]
    output_path_set = [output_path_L, output_path_M, output_path_H]

    for level, (array, output_path) in enumerate(zip(raster_set, output_path_set)):
        if len(supplement_codes):
            array += supplement_demand[level].sum(axis=0)
            write_economy_supplement(output_path.with_name(output_path.stem + '-supplement.tif'),
                                     supplement_demand[level], economy_raster_.profile, supplement_codes)
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=economy_raster_.shape[0],
            width=economy_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=economy_raster_.crs,
            transform=economy_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(array, 1)


In [ ]:
def add_extraction_demand_together(material, scenario, res='50km'):
    RESULTS = ROOT / "results"
    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    country_raster = country_raster_.read(1)
    supplement_codes, _ = economy_supplement(res)
    supplement_totals = np.zeros((3, len(supplement_codes), *country_raster_.shape), dtype=np.float32)
    total_demand_raster_L = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_M = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_U = np.zeros(shape=country_raster_.shape, dtype="float32")

    for start, end in zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050]):
        input_path_l = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-L.tif'
        input_path_m = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-M.tif'
        input_path_u = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-H.tif'

        total_demand_raster_L += rio.open(input_path_l).read(1)
        total_demand_raster_M += rio.open(input_path_m).read(1)
        total_demand_raster_U += rio.open(input_path_u).read(1)
        if len(supplement_codes):
            for level, path in enumerate([input_path_l, input_path_m, input_path_u]):
                with rio.open(path.with_name(path.stem + '-supplement.tif')) as src:
                    if src.descriptions != tuple(str(int(c)) for c in supplement_codes):
                        raise ValueError('Supplement economy bands do not match')
                    supplement_totals[level] += src.read()
    
    global_l = np.sum(total_demand_raster_L) / 1e9
    global_m = np.sum(total_demand_raster_M) / 1e9
    global_u = np.sum(total_demand_raster_U) / 1e9

    print(f'{material} ({scenario}): {global_m:.0f} ({global_l:.0f}-{global_u:.0f}) Gt')

    output_path_l = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-L.tif'
    output_path_m = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-M.tif'
    output_path_u = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-H.tif'

    for level, (total_demand_raster, output_path) in enumerate(zip(
        [total_demand_raster_L, total_demand_raster_M, total_demand_raster_U],
        [output_path_l, output_path_m, output_path_u])):
        if len(supplement_codes):
            write_economy_supplement(output_path.with_name(output_path.stem + '-supplement.tif'),
                                     supplement_totals[level], country_raster_.profile, supplement_codes)
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=country_raster_.shape[0],
            width=country_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=country_raster_.crs,
            transform=country_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(total_demand_raster, 1)

In [ ]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['BAU', 'CE']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

In [ ]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['MIU', 'LE', 'RE', 'RC', 'SS', 'CCR']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

In [ ]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['CE10', 'CE20', 'CE30', 'CE40', 'CE50', 'CE60', 'CE70', 'CE80', 'CE90']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

In [ ]:
for scenario in tqdm(['CE10', 'CE20', 'CE30', 'CE40', 'CE50', 'CE60', 'CE70', 'CE80', 'CE90']):
    add_extraction_demand_together('sand', scenario, res='50km')
    add_extraction_demand_together('gravel', scenario, res='50km')
    add_extraction_demand_together('limestone', scenario, res='50km')

In [ ]:
for scenario in tqdm(['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']):
    add_extraction_demand_together('sand', scenario, res='50km')
    add_extraction_demand_together('gravel', scenario, res='50km')
    add_extraction_demand_together('limestone', scenario, res='50km')